<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/7_Reto_Elasticsearch_Noticias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir reto en Google Colab"></a>

# Reto de transferencia S07 — Radar de noticias

Ya construiste un buscador sobre procesos contractuales. Ahora demuestra que aprendiste **Elasticsearch y no solo el caso**.

## Producto

Una mesa editorial necesita recuperar rápidamente las **5 noticias más relevantes** para una necesidad informativa.

Vas a construir un segundo índice con **126 noticias públicas de agosto de 2026**, usadas previamente en el curso, y entregar:

- un ranking A;
- un ranking B después de cambiar solo el peso del título;
- una medida de calidad <code>Precision@5</code>;
- una recomendación sobre cuál configuración conservar.

**No hay datos de contratación en este reto.** El dominio cambia; las herramientas no.

## Reglas del reto

1. Usa el mismo proyecto Elasticsearch que ya tienes.
2. Crea un índice diferente; no reutilices el índice contractual.
3. No copies API keys dentro del notebook.
4. Mantén constante corpus, consulta y número de resultados.
5. Entre A y B cambia **solo el peso de los campos**.
6. Evalúa relevancia usando el criterio asignado antes de decidir cuál ranking es mejor.

### Datos

Fuente versionada del curso:

<code>Datos/noticias_eltiempo_2026-08.json</code>

Metadatos del archivo indican 126 documentos recuperados de fuentes públicas de EL TIEMPO para uso docente.

---
# 1 · Cargar y preparar las noticias

El JSON conserva estructuras ricas: etiquetas, imágenes y bloques del cuerpo. Para búsqueda no necesitamos indexar todo.

Crearemos un documento más pequeño con:

- <code>titulo</code>
- <code>subtitulo</code>
- <code>texto</code>
- <code>categoria</code>
- <code>seccion</code>
- <code>publicado</code>
- <code>premium</code>
- <code>url</code>

In [ ]:
import pandas as pd
import json, re, unicodedata
from pathlib import Path
from IPython.display import display

URL_NOTICIAS = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/noticias_eltiempo_2026-08.json"

noticias_raw = pd.read_json(URL_NOTICIAS)
print("Noticias cargadas:", len(noticias_raw))
display(
    noticias_raw[["titulo","categoria","seccion","publicado","premium"]]
    .head(8)
)

### Limpiar el cuerpo

El campo <code>cuerpo</code> incluye imágenes, HTML, enlaces y texto editorial.

Para el buscador conservaremos solo bloques textuales útiles. Excluimos HTML incrustado para evitar indexar CSS, botones o fragmentos de interfaz como si fueran contenido periodístico.

In [ ]:
TIPOS_TEXTO = {"title","subtitle","paragraph","subtitle-h2","extra-subtitle"}

def extraer_texto_cuerpo(bloques):
    piezas = []
    if not isinstance(bloques, list):
        return ""
    for bloque in bloques:
        if not isinstance(bloque, dict):
            continue
        if bloque.get("tipo") in TIPOS_TEXTO and bloque.get("texto"):
            piezas.append(str(bloque["texto"]).strip())
    return " ".join(piezas)

noticias = pd.DataFrame({
    "id_noticia": noticias_raw["_id"].astype(str),
    "titulo": noticias_raw["titulo"].fillna("").astype(str),
    "subtitulo": noticias_raw.get("subtitulo", "").fillna("").astype(str),
    "texto": noticias_raw["cuerpo"].map(extraer_texto_cuerpo),
    "categoria": noticias_raw["categoria"].fillna("Sin categoría").astype(str),
    "seccion": noticias_raw["seccion"].fillna("sin-seccion").astype(str),
    "publicado": noticias_raw["publicado"].astype(str),
    "premium": noticias_raw["premium"].astype(bool),
    "url": noticias_raw["url"].astype(str)
})

print("Documentos listos:", len(noticias))
print("Categorías:", noticias["categoria"].nunique())
display(noticias[["titulo","categoria","texto"]].head(5))

### Lectura rápida del corpus

Antes de indexar, mira la distribución por categoría. Elasticsearch no corrige una mala comprensión de los datos de entrada.

In [ ]:
categorias = (
    noticias["categoria"]
    .value_counts()
    .rename_axis("categoria")
    .reset_index(name="n")
)
display(categorias)

---
# 2 · Conectar al mismo proyecto Elasticsearch

Usa la misma **Project URL** y una API key válida. Crearemos un índice aislado por alias.

In [ ]:
!pip -q install -U elasticsearch pandas tabulate

In [ ]:
from getpass import getpass
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

ALIAS = "equipo_demo" #@param {type:"string"}

def slug(valor):
    valor = unicodedata.normalize("NFKD", str(valor))
    valor = "".join(c for c in valor if not unicodedata.combining(c)).lower()
    valor = re.sub(r"[^a-z0-9]+", "-", valor).strip("-")
    return (valor or "equipo-demo")[:35]

alias_seguro = slug(ALIAS)
INDEX_NAME = f"s07-noticias-{alias_seguro}"

endpoint = input("Project URL de Elasticsearch: ").strip()
api_key = getpass("API key: ").strip()

client = Elasticsearch(endpoint, api_key=api_key, request_timeout=30)
info = client.info()

print("Conexión verificada.")
print("Índice del reto:", INDEX_NAME)

---
# 3 · Diseñar el mapping

Aquí el dominio cambió, por lo tanto también cambian los campos.

| Campo | Uso | Tipo |
|---|---|---|
| título | full-text, muy informativo | <code>text</code> |
| subtítulo | full-text | <code>text</code> |
| texto | full-text extenso | <code>text</code> |
| categoría | filtro exacto | <code>keyword</code> |
| sección | filtro exacto | <code>keyword</code> |
| publicado | rango temporal | <code>date</code> |
| premium | filtro booleano | <code>boolean</code> |
| URL | recuperar, no buscar | <code>keyword</code> no indexado |

In [ ]:
mappings = {
    "properties": {
        "id_noticia": {"type":"keyword"},
        "titulo": {"type":"text","analyzer":"spanish"},
        "subtitulo": {"type":"text","analyzer":"spanish"},
        "texto": {"type":"text","analyzer":"spanish"},
        "categoria": {"type":"keyword"},
        "seccion": {"type":"keyword"},
        "publicado": {"type":"date"},
        "premium": {"type":"boolean"},
        "url": {"type":"keyword","index":False}
    }
}

if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)

client.indices.create(index=INDEX_NAME, mappings=mappings)
print("Índice creado:", INDEX_NAME)

---
# 4 · Ingestar y comprobar

Indexaremos los 126 documentos con <code>bulk()</code>. El ID de Elasticsearch será el ID de la noticia.

In [ ]:
acciones = (
    {
        "_index": INDEX_NAME,
        "_id": d["id_noticia"],
        "_source": d
    }
    for d in noticias.to_dict("records")
)

ok, errores = bulk(
    client,
    acciones,
    refresh=True,
    raise_on_error=False
)

print("Documentos indexados:", ok)
print("Errores:", len(errores))

conteo = client.count(index=INDEX_NAME)["count"]
print("Conteo Elasticsearch:", conteo)
assert conteo == len(noticias) == 126

---
# 5 · Tu necesidad editorial

El alias asigna un reto reproducible.

No cambies la consulta para favorecer tus resultados.

In [ ]:
RETOS = [
    {
        "consulta":"terremoto colombia",
        "criterio":"La noticia trata directamente el terremoto ocurrido en Colombia, sus impactos, respuesta, reconstrucción o explicación científica."
    },
    {
        "consulta":"salud publica",
        "criterio":"La noticia trata directamente instituciones, políticas, riesgos, servicios o acciones de salud pública."
    },
    {
        "consulta":"tecnologia inteligencia artificial",
        "criterio":"La noticia aborda tecnología digital o inteligencia artificial como tema central, no como mención incidental."
    },
    {
        "consulta":"futbol colombia",
        "criterio":"La noticia trata fútbol colombiano, equipos colombianos o participación colombiana en competiciones de fútbol."
    },
    {
        "consulta":"educacion colombia",
        "criterio":"La noticia aborda educación, instituciones educativas, estudiantes, docentes o políticas educativas en Colombia."
    }
]

reto = RETOS[sum(ord(c) for c in alias_seguro) % len(RETOS)]
CONSULTA = reto["consulta"]
CRITERIO = reto["criterio"]

print("Consulta asignada:", CONSULTA)
print("Criterio de relevancia:", CRITERIO)

---
# 6 · Configuración A

Todos los campos textuales tienen el mismo peso.

La función también solicita <code>highlight</code> para que puedas inspeccionar por qué apareció cada noticia.

In [ ]:
def ejecutar_busqueda(campos, categoria=None):
    bool_query = {
        "must": [{
            "multi_match": {
                "query": CONSULTA,
                "fields": campos
            }
        }]
    }
    if categoria:
        bool_query["filter"] = [{"term":{"categoria":categoria}}]

    return client.search(
        index=INDEX_NAME,
        query={"bool": bool_query},
        highlight={
            "fields":{
                "titulo":{},
                "subtitulo":{},
                "texto":{"fragment_size":180,"number_of_fragments":1}
            }
        },
        size=5
    )

def tabla_hits(resp):
    filas=[]
    for rank,h in enumerate(resp["hits"]["hits"],start=1):
        s=h["_source"]
        frag=[]
        for parts in h.get("highlight",{}).values():
            frag.extend(parts)
        filas.append({
            "rank":rank,
            "id_noticia":s["id_noticia"],
            "score":h["_score"],
            "titulo":s["titulo"],
            "categoria":s["categoria"],
            "publicado":s["publicado"],
            "fragmento":" ... ".join(frag),
            "url":s["url"]
        })
    return pd.DataFrame(filas)

resp_A = ejecutar_busqueda(["titulo","subtitulo","texto"])
tabla_A = tabla_hits(resp_A)
display(tabla_A)

---
# 7 · Configuración B

Ahora cambia **una sola decisión**:

- título × 4;
- subtítulo × 2;
- cuerpo × 1.

La hipótesis es que una coincidencia en el título debe tener mayor peso editorial que una mención perdida en el cuerpo.

In [ ]:
resp_B = ejecutar_busqueda(["titulo^4","subtitulo^2","texto"])
tabla_B = tabla_hits(resp_B)
display(tabla_B)

## ¿Qué se movió?

Compara posiciones de los IDs, no solo scores.

In [ ]:
comparacion = (
    tabla_A[["id_noticia","rank"]]
    .rename(columns={"rank":"rank_A"})
    .merge(
        tabla_B[["id_noticia","rank"]].rename(columns={"rank":"rank_B"}),
        on="id_noticia",
        how="outer"
    )
)
comparacion["cambio"] = comparacion["rank_A"] - comparacion["rank_B"]
display(comparacion.sort_values(["rank_A","rank_B"], na_position="last"))

---
# 8 · Evaluar la calidad: Precision@5

No decidas que B es mejor solo porque cambió.

Usa el criterio que ya estaba fijado **antes de etiquetar**:

> **Criterio:** <span id="criterio"></span>

Para cada resultado marca:

- <code>1</code> si responde al criterio;
- <code>0</code> si no.

In [ ]:
def etiquetar(tabla, nombre):
    etiquetas=[]
    print(f"\n=== {nombre} ===")
    print("Consulta:", CONSULTA)
    print("Criterio:", CRITERIO)
    for _,fila in tabla.iterrows():
        print("\nRank", int(fila["rank"]), "·", fila["titulo"])
        print("Fragmento:", str(fila["fragmento"])[:500])
        while True:
            x=input("¿Relevante? [1/0]: ").strip()
            if x in {"0","1"}:
                etiquetas.append(int(x))
                break
            print("Escribe 1 o 0.")
    return etiquetas

et_A = etiquetar(tabla_A,"A")
et_B = etiquetar(tabla_B,"B")

p5_A = sum(et_A)/5
p5_B = sum(et_B)/5

print("\nP@5 A:", p5_A)
print("P@5 B:", p5_B)
print("Δ P@5:", round(p5_B-p5_A,3))

### Cómo interpretar

- Si B cambia posiciones y mejora P@5, tienes evidencia local de que el boost ayudó para esta consulta.
- Si B cambia posiciones pero P@5 no mejora, “más tuning” no significa “mejor búsqueda”.
- Una sola consulta y cinco juicios **no** validan un buscador de producción.

El paso profesional sería ampliar el conjunto de consultas y ratings y usar herramientas de evaluación como <code>_rank_eval</code>.

---
# 9 · Reto adicional: filtro exacto

El editor ahora pide restringir la búsqueda a una categoría.

Elige una categoría **solo si tiene sentido para tu necesidad**. Si no, deja el texto vacío.

In [ ]:
CATEGORIA = "" #@param {type:"string"}

if CATEGORIA.strip():
    if CATEGORIA not in set(noticias["categoria"]):
        print("Categoría no encontrada. Revisa la tabla de categorías del inicio.")
    else:
        filtrada = tabla_hits(
            ejecutar_busqueda(
                ["titulo^4","subtitulo^2","texto"],
                categoria=CATEGORIA
            )
        )
        display(filtrada)
else:
    print("Sin filtro de categoría.")

---
# 10 · Entrega del radar editorial

La entrega debe ser pequeña y reproducible:

- ranking A y B;
- consulta y criterio;
- P@5 A y B;
- una recomendación entre A y B;
- un resultado dudoso y por qué;
- una limitación concreta.

In [ ]:
RECOMENDACION = "Elige A o B y justifica con P@5 y un cambio de ranking." #@param {type:"string"}
DUDOSO = "Identifica un resultado dudoso y explica por qué apareció." #@param {type:"string"}
LIMITE = "Escribe una limitación concreta de tu evaluación." #@param {type:"string"}

salida = pd.concat([
    tabla_A.assign(configuracion="A", relevante=et_A),
    tabla_B.assign(configuracion="B", relevante=et_B)
], ignore_index=True)
salida["consulta"] = CONSULTA
salida.to_csv("s07_reto_noticias_resultados.csv", index=False, encoding="utf-8-sig")

config = {
    "alias": alias_seguro,
    "index_name": INDEX_NAME,
    "documentos": len(noticias),
    "consulta": CONSULTA,
    "criterio": CRITERIO,
    "config_A":["titulo","subtitulo","texto"],
    "config_B":["titulo^4","subtitulo^2","texto"],
    "precision_at_5_A":p5_A,
    "precision_at_5_B":p5_B,
    "categoria_filtro": CATEGORIA.strip() or None
}
Path("s07_reto_noticias_config.json").write_text(
    json.dumps(config,ensure_ascii=False,indent=2),
    encoding="utf-8"
)

informe=f"""# Reto S07 · Radar de noticias

- Alias: {alias_seguro}
- Documentos: {len(noticias)}
- Consulta: {CONSULTA}
- Criterio: {CRITERIO}
- P@5 A: {p5_A}
- P@5 B: {p5_B}

## Recomendación
{RECOMENDACION}

## Resultado dudoso
{DUDOSO}

## Limitación
{LIMITE}
"""
Path("s07_reto_noticias.md").write_text(informe,encoding="utf-8")

print("Archivos creados:")
print("- s07_reto_noticias_resultados.csv")
print("- s07_reto_noticias_config.json")
print("- s07_reto_noticias.md")

In [ ]:
try:
    from google.colab import files
    for archivo in [
        "s07_reto_noticias_resultados.csv",
        "s07_reto_noticias_config.json",
        "s07_reto_noticias.md"
    ]:
        files.download(archivo)
except ImportError:
    print("Fuera de Colab: los archivos quedaron en el directorio de trabajo.")

## Cierre

El dominio cambió de contratos a noticias, pero la arquitectura permaneció:

**documento → mapping → analyzer → bulk → query → ranking → evaluación**

Si pudiste trasladar esas decisiones sin copiar el ejemplo contractual, el aprendizaje ya es transferible.